# 00 QA — 환경 검증

학습을 시작하기 전에 모든 의존성과 핵심 로직이 올바르게 동작하는지 확인합니다.

모든 셀이 오류 없이 실행되고 마지막 셀에 `✅ All checks passed` 가 출력되면 준비 완료입니다.

In [ ]:
# ── 1. 환경 정보 ──────────────────────────────────────────────
import sys, torch, torchvision, accelerate as acc_lib
import numpy, sklearn, pandas, yaml

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"accelerate  : {acc_lib.__version__}")
print(f"CUDA 사용가능: {torch.cuda.is_available()}")
n_gpu = torch.cuda.device_count()
print(f"GPU 수      : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")


In [ ]:
# ── 2. 프로젝트 import 전체 검증 ─────────────────────────────
from src.models.unet import UNetModel
from src.models.diffusion import Diffusion
from src.sdd.distillation import Center, dino_loss
from src.sdd.gating import timestep_gate
from src.sdd.ema import clone_model, ema_update
from src.sdd.projection_head import ProjectionHead
from src.training.trainer import SDDTrainer
from src.training.losses import total_loss
from src.evaluation.feature_extract import extract_features
from src.evaluation.fid import FIDEvaluator
from src.evaluation.linear_probe import train_linear_probe
from src.experiments import (
    load_cfg, deep_update, build_model, build_trainer,
    launch_train, launch_eval_fid, launch_eval_linear,
)
print("✓ 모든 import 성공")


In [ ]:
# ── 3. UNet forward shape 검증 ────────────────────────────────
import torch
model = UNetModel(base_channels=32, channel_mults=(1, 2), num_res_blocks=1,
                  attention_resolutions=(8,), image_size=16)
x = torch.randn(2, 3, 16, 16)
t = torch.randint(0, 100, (2,))
out = model(x, t)
assert out.shape == x.shape, f"출력 shape 불일치: {out.shape}"
print(f"✓ UNet forward: 입력 {tuple(x.shape)} → 출력 {tuple(out.shape)}")

for layer in ["bottleneck", "skip1", "skip2", "decoder1"]:
    out2, feat = model(x, t, return_features=True, feature_layer=layer)
    assert out2.shape == x.shape
    assert feat.ndim == 2 and feat.shape[0] == 2
    print(f"  ✓ feature_layer='{layer}'  feat.shape={tuple(feat.shape)}")


In [ ]:
# ── 4. Diffusion q_sample z ───────────────────────────────
from src.models.diffusion import Diffusion
diffusion = Diffusion(timesteps=100, beta_schedule="linear")
x0 = torch.randn(4, 3, 16, 16)
t  = torch.randint(0, 100, (4,))
xt, noise = diffusion.q_sample(x0, t)
assert xt.shape == x0.shape
assert (diffusion.alphas_cumprod[1:] <= diffusion.alphas_cumprod[:-1]).all(), "alphas_cumprod 단조감소 실패"
print(f"✓ Diffusion q_sample: {tuple(xt.shape)}")
print(f"✓ alphas_cumprod 단조감소 확인")


In [ ]:
# ── 5. Gating 로직 검증 ──────────────────────────────────────
from src.sdd.gating import timestep_gate
T = 1000
t_all = torch.arange(T)

g_hard = timestep_gate(t_all, T, mode="hard", t_min=0.1, t_max=0.6)
assert set(g_hard.tolist()) <= {0.0, 1.0}, "hard gate가 이진값이 아님"
t_norm = t_all.float() / (T - 1)
inside  = (t_norm >= 0.1) & (t_norm <= 0.6)
assert g_hard[inside].all()
assert (g_hard[~inside] == 0).all()
print(f"✓ hard gate: {inside.sum().item()} / {T} 스텝 활성화")

g_soft = timestep_gate(t_all, T, mode="soft", t_min=0.1, t_max=0.6, soft_beta=0.08)
assert (g_soft >= 0).all() and (g_soft <= 1).all(), "soft gate 범위 초과"
print(f"✓ soft gate: 평균값 = {g_soft.mean():.3f}")

g_none = timestep_gate(t_all, T, mode="none")
assert (g_none == 1.0).all()
print(f"✓ none gate: 전부 1.0")


In [ ]:
# ── 6. Distillation loss 검증 ────────────────────────────────
from src.sdd.distillation import Center, dino_loss
B, D = 8, 64
teacher = torch.randn(B, D)
student = torch.randn(B, D)

loss_rand = dino_loss(teacher, student, center=None, use_centering=False)
assert float(loss_rand) >= 0, "DINO loss가 음수"

loss_match = dino_loss(teacher, teacher.clone(), center=None, use_centering=False)
assert float(loss_match) < float(loss_rand), "일치하는 student의 loss가 더 크거나 같음"
print(f"✓ dino_loss: random={float(loss_rand):.4f}, matched={float(loss_match):.4f}")

center = Center(D, momentum=0.9)
for _ in range(30):
    center.update(torch.ones(B, D))
assert center().mean() > 0.9
print(f"✓ Center momentum 수렴: {center().mean():.4f}")


In [ ]:
# ── 7. EMA 업데이트 수식 검증 ────────────────────────────────
from src.sdd.ema import clone_model, ema_update

student = UNetModel(base_channels=16, channel_mults=(1,), num_res_blocks=1,
                    attention_resolutions=(), image_size=8)
teacher = clone_model(student)

# teacher 파라미터가 gradient 없는지 확인
for p in teacher.parameters():
    assert not p.requires_grad, "teacher에 gradient 활성화됨"

# 수식: t_new = m * t_old + (1 - m) * s
pname = list(dict(teacher.named_parameters()).keys())[0]
t_before = dict(teacher.named_parameters())[pname].data.clone()
with torch.no_grad():
    dict(student.named_parameters())[pname].data.fill_(1.0)
m = 0.9
ema_update(teacher, student, momentum=m)
t_after = dict(teacher.named_parameters())[pname].data
expected = t_before * m + 1.0 * (1 - m)
assert torch.allclose(t_after, expected, atol=1e-5), "EMA 수식 오류"
print(f"✓ EMA 업데이트 수식 확인 (momentum={m})")


In [ ]:
# ── 8. total_loss 검증 ───────────────────────────────────────
from src.training.losses import total_loss

eps_pred  = torch.randn(4, 3, 8, 8)
noise     = torch.randn(4, 3, 8, 8)
s_logits  = torch.randn(4, 64)
t_logits  = torch.randn(4, 64)
center    = Center(64)
timesteps = torch.randint(0, 100, (4,))

# lambda_distill=0 → distill loss = 0
_, m0 = total_loss(eps_pred, noise, s_logits, t_logits, center, timesteps, 100, lambda_distill=0.0)
assert m0["loss_distill"] == 0.0
print("✓ lambda_distill=0 → loss_distill=0")

# full loss > 0
loss_full, mf = total_loss(eps_pred, noise, s_logits, t_logits, center, timesteps, 100,
                           lambda_distill=0.5, use_gating=True, gating_mode="hard",
                           t_min=0.1, t_max=0.6)
assert float(loss_full) > 0
assert 0.0 <= mf["gate_mean"] <= 1.0
print(f"✓ full loss = {float(loss_full):.4f}, gate_mean = {mf['gate_mean']:.3f}")

# 게이팅 OFF 구간에서 distill=0 확인
t_outside = torch.zeros(4, dtype=torch.long)   # t=0 → t_norm=0 < t_min=0.1
_, m_out = total_loss(eps_pred, noise, s_logits, t_logits, center, t_outside, 100,
                      lambda_distill=0.5, use_gating=True, gating_mode="hard",
                      t_min=0.1, t_max=0.6)
assert m_out["gate_mean"] == 0.0, "게이트 외부 timestep에서 gate_mean != 0"
print("✓ 게이트 범위 밖 timestep → gate_mean=0.0")


In [ ]:
# ── 9. notebook_launcher 동작 확인 ───────────────────────────
from accelerate import notebook_launcher

results = []

def _dummy_fn(x, y):
    import torch
    from accelerate import Accelerator
    acc = Accelerator()
    if acc.is_main_process:
        results.append(x + y)

notebook_launcher(_dummy_fn, args=(3, 4), num_processes=1)
assert results == [7], f"notebook_launcher 결과 오류: {results}"
print(f"✓ notebook_launcher(num_processes=1): 3 + 4 = {results[0]}")


In [ ]:
# ── 10. config 로드 검증 ─────────────────────────────────────
from src.experiments import load_cfg, deep_update

cfg = load_cfg("configs/cifar10.yaml")
required_keys = ["dataset", "train", "model", "diffusion", "sdd", "output"]
for k in required_keys:
    assert k in cfg, f"config에 '{k}' 키 없음"

cfg2 = deep_update(cfg, {"train": {"epochs": 999}})
assert cfg2["train"]["epochs"] == 999
assert cfg["train"]["epochs"] != 999   # 원본 불변
print("✓ load_cfg / deep_update 동작 확인")


In [ ]:
# ── 최종 결과 ─────────────────────────────────────────────────
print()
print("=" * 50)
print("✅  All checks passed — 학습을 시작할 수 있습니다.")
print("=" * 50)
